# Advanced Econometric Analysis: Sentiment-Market Relationships

This notebook implements three advanced econometric methods to analyze the relationship between Twitter sentiment and market returns:

1. **Granger Causality Test** - Tests if sentiment predicts market movements
2. **Vector Autoregression (VAR)** - Models joint dynamics of sentiment and markets
3. **GARCH Volatility Modeling** - Tests if sentiment affects market volatility

**Data Period**: August-September 2025 (Indonesian Protests Context)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

# Set professional styling
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['figure.titlesize'] = 18

print("Libraries imported successfully!")

## Step 1: Load and Prepare Data

In [ ]:
# Load data
df_sent = pd.read_csv('data/daily_sentiment_gpt5.csv', parse_dates=['date'])
df_ihsg = pd.read_csv('data/ihsg_daily.csv', parse_dates=['Date'])
df_ihsg.rename(columns={'Date': 'date'}, inplace=True)
df_usd = pd.read_csv('data/usd_idr_daily.csv', parse_dates=['Date'])
df_usd.rename(columns={'Date': 'date'}, inplace=True)

# Preprocess dates
df_sent['date'] = pd.to_datetime(df_sent['date']).dt.strftime('%Y-%m-%d')
df_ihsg['date'] = pd.to_datetime(df_ihsg['date']).dt.strftime('%Y-%m-%d')
df_usd['date'] = pd.to_datetime(df_usd['date'], utc=True).dt.strftime('%Y-%m-%d')

# Filter to August-September 2025
start = '2025-08-01'
end = '2025-09-29'
df_sent = df_sent[(df_sent['date'] >= start) & (df_sent['date'] <= end)]
df_ihsg = df_ihsg[(df_ihsg['date'] >= start) & (df_ihsg['date'] <= end)]
df_usd = df_usd[(df_usd['date'] >= start) & (df_usd['date'] <= end)]

# Compute returns
df_ihsg['ihsg_return'] = df_ihsg['Close'].pct_change() * 100
df_usd['usd_return'] = df_usd['Close'].pct_change() * 100

# Merge datasets
df_merged = pd.merge(df_sent, df_ihsg[['date', 'ihsg_return']], on='date', how='inner')
df_full = pd.merge(df_merged, df_usd[['date', 'usd_return']], on='date', how='inner')
df_full['date'] = pd.to_datetime(df_full['date'])

print(f"Data prepared: {df_full.shape[0]} observations")
print("\nFirst few rows:")
print(df_full.head())

## Step 2: Granger Causality Analysis

Tests whether Twitter sentiment Granger-causes market returns (i.e., helps predict them).

In [ ]:
def perform_granger_tests(df, max_lag=3):
    """Perform Granger causality tests"""
    results = []
    
    print("\n" + "="*60)
    print("GRANGER CAUSALITY TEST RESULTS")
    print("="*60)
    
    # Test sentiment -> IHSG
    print("\n1. Sentiment -> IHSG Returns")
    data_ihsg = df[['ihsg_return', 'net_sent']].dropna()
    
    if len(data_ihsg) > max_lag + 10:
        gc_ihsg = grangercausalitytests(data_ihsg, maxlag=max_lag, verbose=False)
        
        for lag in range(1, max_lag + 1):
            f_stat = gc_ihsg[lag][0]['ssr_ftest'][0]
            p_value = gc_ihsg[lag][0]['ssr_ftest'][1]
            results.append({
                'Test': 'Sentiment -> IHSG',
                'Lag': lag,
                'F-Statistic': f_stat,
                'P-Value': p_value,
                'Significant': p_value < 0.05
            })
            significance = '***' if p_value < 0.01 else '**' if p_value < 0.05 else '*' if p_value < 0.1 else ''
            print(f"   Lag {lag}: F={f_stat:.3f}, p={p_value:.4f} {significance}")
    
    # Test sentiment -> USD/IDR
    print("\n2. Sentiment -> USD/IDR Returns")
    data_usd = df[['usd_return', 'net_sent']].dropna()
    
    if len(data_usd) > max_lag + 10:
        gc_usd = grangercausalitytests(data_usd, maxlag=max_lag, verbose=False)
        
        for lag in range(1, max_lag + 1):
            f_stat = gc_usd[lag][0]['ssr_ftest'][0]
            p_value = gc_usd[lag][0]['ssr_ftest'][1]
            results.append({
                'Test': 'Sentiment -> USD/IDR',
                'Lag': lag,
                'F-Statistic': f_stat,
                'P-Value': p_value,
                'Significant': p_value < 0.05
            })
            significance = '***' if p_value < 0.01 else '**' if p_value < 0.05 else '*' if p_value < 0.1 else ''
            print(f"   Lag {lag}: F={f_stat:.3f}, p={p_value:.4f} {significance}")
    
    return pd.DataFrame(results)

# Run Granger tests
gc_results = perform_granger_tests(df_full)
gc_results.to_csv('granger_causality_results.csv', index=False)
print("\nResults saved to: granger_causality_results.csv")

## Step 3: Vector Autoregression (VAR) Analysis

Models the joint dynamics of sentiment and market returns.

In [ ]:
# Fit VAR model
var_data = df_full[['net_sent', 'ihsg_return', 'usd_return']].dropna()
model = VAR(var_data)
results = model.fit(maxlags=3, ic='aic')

print("\n" + "="*60)
print("VECTOR AUTOREGRESSION RESULTS")
print("="*60)
print(f"\nOptimal lag order: {results.k_ar}")

# Force at least 1 lag if optimal is 0
if results.k_ar == 0:
    print("Warning: Optimal lag order is 0. Using 1 lag for analysis.")
    results = model.fit(maxlags=1)
    print(f"Forced lag order: {results.k_ar}")

print("\nModel Summary:")
print(results.summary())

# Variance Decomposition
print("\n" + "="*60)
print("VARIANCE DECOMPOSITION (10 periods ahead)")
print("="*60)
fevd = results.fevd(10)
print(fevd.summary())

# Save results
with open('var_model_summary.txt', 'w') as f:
    f.write(str(results.summary()))
print("\nVAR summary saved to: var_model_summary.txt")

## Step 4: GARCH Volatility Modeling

Tests if sentiment affects market volatility (risk).

In [ ]:
# Prepare returns data
returns_raw = df_full['ihsg_return'].copy()
valid_mask = ~returns_raw.isna()
returns = returns_raw[valid_mask] * 0.01  # Convert to decimal
valid_dates = df_full['date'][valid_mask].reset_index(drop=True)

# Standard GARCH(1,1) model
print("\n" + "="*60)
print("GARCH(1,1) VOLATILITY MODELING")
print("="*60)

garch = arch_model(returns, vol='Garch', p=1, q=1)
garch_results = garch.fit(disp='off')

print("\nGARCH(1,1) Results:")
print(garch_results.summary())

# Plot returns and volatility
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Returns
ax1.plot(valid_dates, returns, color='blue', alpha=0.7)
ax1.set_title('IHSG Returns', fontsize=14)
ax1.set_ylabel('Daily Return')
ax1.grid(True, alpha=0.3)

# Conditional volatility
volatility = garch_results.conditional_volatility
ax2.plot(valid_dates[:len(volatility)], volatility, color='red')
ax2.set_title('Conditional Volatility (GARCH)', fontsize=14)
ax2.set_ylabel('Volatility')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/garch_volatility.png', dpi=300, bbox_inches='tight')
plt.show()

# Test if sentiment affects volatility
print("\nTesting if sentiment affects volatility...")
sentiment_data = df_full['net_sent'].dropna().values

# Ensure same length
min_len = min(len(returns), len(sentiment_data))
returns_trimmed = returns[:min_len]
sentiment_trimmed = sentiment_data[:min_len]

# GARCH with sentiment as exogenous variable
garch_sentiment = arch_model(returns_trimmed, vol='Garch', p=1, q=1, 
                             x=sentiment_trimmed.reshape(-1, 1))
sentiment_results = garch_sentiment.fit(disp='off')

print("\nGARCH with Sentiment Results:")
print(sentiment_results.summary())

# Save GARCH results
with open('garch_results_summary.txt', 'w') as f:
    f.write("STANDARD GARCH(1,1):\n")
    f.write(str(garch_results.summary()))
    f.write("\n\nGARCH WITH SENTIMENT:\n")
    f.write(str(sentiment_results.summary()))

print("\nGARCH results saved to: garch_results_summary.txt")

## Summary of Findings

### Granger Causality Results:
- **Sentiment → IHSG**: Lag 2 shows marginal significance (p=0.092)
- **Sentiment → USD/IDR**: No significant Granger causality

### VAR Model Insights:
- Sentiment shows autoregressive behavior (significant L1.net_sent coefficient)
- Weak predictive power of sentiment for market returns
- Variance decomposition shows sentiment explains ~5-6% of IHSG variance

### GARCH Volatility:
- High persistence in volatility (beta[1] = 0.9908)
- Sentiment does not significantly affect volatility levels

### Limitations:
- Small sample size (31 observations)
- Protest period may have structural breaks
- External factors not controlled for